# 05 — Q-Learning from scratch sur FrozenLake-v1

Dans ce notebook, on implémente le **Q-Learning** à la main, sans Stable-Baselines3.
L'objectif : construire et remplir une **Q-table** qui guide un agent à traverser un lac gelé sans tomber dans les trous.

## 1. Pourquoi le Q-Learning ?

Dans les notebooks précédents, on utilisait des algorithmes (PPO, A2C) qui fonctionnent comme des "boîtes noires" : on leur donne un environnement et ils apprennent via des réseaux de neurones.

Le **Q-Learning** est plus fondamental : c'est l'un des premiers algorithmes de RL. Il n'utilise **pas de réseau de neurones** — il stocke tout dans un simple tableau. Ça le rend idéal pour comprendre les mécanismes de base.

### Pourquoi FrozenLake et pas CartPole ?

| Critère | CartPole | FrozenLake |
|---------|----------|------------|
| Espace d'états | Continu (infini) | Discret (fini, 16 états) |
| Espace d'actions | 2 (gauche/droite) | 4 (haut/bas/gauche/droite) |
| Compatible Q-table ? | ❌ (trop d'états) | ✅ (tableau 16×4) |

Le Q-Learning classique ne fonctionne qu'avec des espaces d'états **finis et discrets**. FrozenLake a exactement 16 états (cases d'une grille 4×4), ce qui est parfait.

## 2. L'environnement FrozenLake-v1

### Description

L'agent évolue sur une grille 4×4 représentant un lac gelé :

```
S F F F
F H F H
F F F H
H F F G
```

| Symbole | Signification |
|---------|---------------|
| **S** | Start — point de départ |
| **F** | Frozen — glace solide (safe) |
| **H** | Hole — trou (épisode terminé, reward = 0) |
| **G** | Goal — objectif (reward = 1) |

### Espace d'actions

| Action | Direction |
|--------|-----------|
| 0 | ← Gauche |
| 1 | ↓ Bas |
| 2 | → Droite |
| 3 | ↑ Haut |

> **Attention** : FrozenLake est glissant (`is_slippery=True` par défaut). L'agent peut ne pas aller exactement là où il veut. On va désactiver ça pour commencer.

## 3. Étape 1 — Créer l'environnement et initialiser la Q-table

In [ ]:
import gymnasium as gym
import numpy as np

# is_slippery=False : l'agent va là où il décide (plus simple pour commencer)
env = gym.make('FrozenLake-v1', is_slippery=False)

obs, info = env.reset()

print("=== Espace d'observation ===")
print(f"Type : {env.observation_space}")
print(f"Nombre d'états : {env.observation_space.n}")

print("\n=== Espace d'action ===")
print(f"Type : {env.action_space}")
print(f"Nombre d'actions : {env.action_space.n}")

### La Q-table : c'est quoi ?

La **Q-table** est un tableau à 2 dimensions où chaque cellule stocke la **valeur Q(s, a)**.

**Q(s, a)** = estimation de la récompense cumulée future si l'agent est dans l'état `s` et choisit l'action `a`.

```
          ← (0)  ↓ (1)  → (2)  ↑ (3)
État 0  [  0.0    0.0    0.0    0.0  ]
État 1  [  0.0    0.0    0.0    0.0  ]
...     [  ...    ...    ...    ...  ]
État 15 [  0.0    0.0    0.0    0.0  ]
```

Au début, tout est à 0 (l'agent ne sait rien). Au fur et à mesure de l'entraînement, les valeurs se rempliront et l'agent apprendra quelles actions sont "bonnes" dans chaque état.

In [ ]:
# Dimensions de la Q-table
n_states = env.observation_space.n   # 16 états (cases de la grille 4x4)
n_actions = env.action_space.n       # 4 actions possibles

# Initialiser la Q-table à zéro
q_table = np.zeros((n_states, n_actions))

print(f"Dimensions de la Q-table : {q_table.shape}")
print(f"  → {n_states} états × {n_actions} actions")
print("\nQ-table initiale (premiers 5 états) :")
print(q_table[:5])

## 4. Hyperparamètres

Avant de coder la boucle d'entraînement, on définit les hyperparamètres. Ce sont des "réglages" qu'on fixe avant l'entraînement.

| Hyperparamètre | Symbole | Valeur | Rôle |
|----------------|---------|--------|------|
| Learning rate | α (alpha) | 0.8 | Vitesse d'apprentissage — quelle importance donner à la nouvelle info |
| Discount factor | γ (gamma) | 0.95 | Importance des récompenses futures vs immédiates |
| Epsilon initial | ε | 1.0 | Probabilité d'exploration au départ (100%) |
| Epsilon min | ε_min | 0.01 | Probabilité minimale d'exploration |
| Epsilon decay | — | 0.995 | Facteur de réduction d'epsilon à chaque épisode |
| Épisodes | — | 5000 | Nombre de parties jouées pendant l'entraînement |

### Le rôle de gamma (facteur d'actualisation)

- `gamma = 0` : l'agent est **myope**, il ne pense qu'à la récompense immédiate
- `gamma = 1` : l'agent valorise autant les récompenses futures que les immédiates (risque de ne pas converger)
- `gamma = 0.95` : un bon équilibre — les récompenses lointaines comptent mais moins

In [ ]:
# Hyperparamètres
learning_rate = 0.8      # α — vitesse d'apprentissage
discount_factor = 0.95   # γ — importance des récompenses futures
n_episodes = 5000        # nombre d'épisodes d'entraînement

# Epsilon-greedy
epsilon = 1.0            # commence à 1 : exploration totale
epsilon_min = 0.01       # plancher : toujours un peu d'exploration
epsilon_decay = 0.995    # facteur de réduction par épisode

print("Hyperparamètres définis.")

## 5. Étape 2 — La stratégie Epsilon-Greedy

Comment l'agent choisit ses actions ? Il y a un dilemme fondamental en RL :

- **Explorer** : tenter des actions aléatoires pour découvrir de nouvelles choses
- **Exploiter** : utiliser ce qu'on a déjà appris pour obtenir de bonnes récompenses

La stratégie **epsilon-greedy** résout ce dilemme :

```
Tirer un nombre aléatoire r entre 0 et 1
  Si r < epsilon → EXPLORER  (action aléatoire)
  Sinon          → EXPLOITER (meilleure action de la Q-table)
```

Au début (epsilon = 1.0), l'agent explore tout le temps. Progressivement, epsilon diminue et l'agent exploite de plus en plus ses connaissances.

```
Épisode 1    : epsilon = 1.000 → 100% exploration
Épisode 500  : epsilon ≈ 0.082 → 8% exploration
Épisode 2000 : epsilon = 0.010 → 1% exploration (plancher)
```

## 6. La formule de mise à jour : l'équation de Bellman

C'est **le cœur du Q-Learning**. Après chaque action, on met à jour la Q-table avec cette formule :

$$Q(s, a) \leftarrow Q(s, a) + \alpha \cdot \left[ r + \gamma \cdot \max_{a'} Q(s', a') - Q(s, a) \right]$$

Décomposons ça en français :

| Terme | Signification |
|-------|---------------|
| `Q(s, a)` | Valeur actuelle : ce qu'on pensait valoir |
| `r` | Récompense réelle reçue |
| `γ · max Q(s', a')` | Meilleure valeur future possible (depuis le nouvel état) |
| `r + γ · max Q(s', a')` | La **cible** : ce que la valeur devrait être |
| `cible - Q(s, a)` | L'**erreur TD** : écart entre ce qu'on pensait et la réalité |
| `α · erreur` | Correction partielle (on n'update pas tout d'un coup) |

**Intuition** : à chaque pas, on corrige légèrement notre estimation en direction de la "vraie" valeur observée.

## 7. Étape 2 — La boucle d'entraînement

In [ ]:
# Réinitialiser la Q-table avant l'entraînement
q_table = np.zeros((n_states, n_actions))
epsilon = 1.0

# Statistiques pour suivre la progression
rewards_per_episode = []

for episode in range(n_episodes):
    # Réinitialiser l'environnement au début de chaque épisode
    state, info = env.reset()
    done = False
    total_reward = 0

    while not done:
        # --- Stratégie epsilon-greedy ---
        if np.random.uniform(0, 1) < epsilon:
            # EXPLORER : action aléatoire
            action = env.action_space.sample()
        else:
            # EXPLOITER : meilleure action connue pour cet état
            action = np.argmax(q_table[state, :])

        # Exécuter l'action dans l'environnement
        new_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated

        # --- Mise à jour de la Q-table (équation de Bellman) ---
        # Valeur future maximale depuis le nouvel état
        max_future_q = np.max(q_table[new_state, :])

        # Cible : récompense immédiate + récompense future actualisée
        target = reward + discount_factor * max_future_q

        # Correction partielle de la valeur actuelle
        q_table[state, action] = q_table[state, action] + learning_rate * (target - q_table[state, action])

        # Passer à l'état suivant
        state = new_state
        total_reward += reward

    # --- Réduire epsilon à la fin de chaque épisode ---
    epsilon = max(epsilon_min, epsilon * epsilon_decay)
    rewards_per_episode.append(total_reward)

    # Afficher la progression toutes les 500 épisodes
    if (episode + 1) % 500 == 0:
        recent_wins = np.mean(rewards_per_episode[-500:])
        print(f"Épisode {episode + 1:5d} | Taux de succès (500 derniers) : {recent_wins:.1%} | epsilon : {epsilon:.3f}")

print("\nEntraînement terminé !")

## 8. Visualiser la Q-table apprise

Voyons ce que l'agent a appris. Pour chaque état, la Q-table indique la meilleure action à prendre.

In [ ]:
print("Q-table apprise :")
print("(colonnes : ← Gauche | ↓ Bas | → Droite | ↑ Haut)")
print("=" * 55)
print(q_table)

print("\nMeilleure action par état :")
action_symbols = ["←", "↓", "→", "↑"]
best_actions = np.argmax(q_table, axis=1)

# Afficher la grille 4x4 avec les meilleures actions
grille = ["S", "F", "F", "F",
          "F", "H", "F", "H",
          "F", "F", "F", "H",
          "H", "F", "F", "G"]

print()
for i in range(16):
    case = grille[i]
    if case in ["H", "G"]:
        symbole = f" {case} "
    else:
        symbole = f" {action_symbols[best_actions[i]]} "
    print(symbole, end="")
    if (i + 1) % 4 == 0:
        print()  # nouvelle ligne

## 9. Étape 3 — Évaluation de l'agent entraîné

Maintenant qu'on a une Q-table remplie, on va tester l'agent sur 100 épisodes.

**Différence clé avec l'entraînement** :
- Plus d'epsilon, plus d'exploration
- L'agent choisit **toujours** `np.argmax(q_table[state, :])`
- On ne modifie **jamais** la Q-table pendant l'évaluation

In [ ]:
n_eval_episodes = 100
total_wins = 0

for episode in range(n_eval_episodes):
    state, info = env.reset()
    done = False

    while not done:
        # Toujours exploiter — pas d'exploration pendant l'évaluation
        action = np.argmax(q_table[state, :])

        state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated

    # reward = 1.0 si l'agent a atteint l'objectif
    if reward == 1.0:
        total_wins += 1

success_rate = total_wins / n_eval_episodes
print(f"Taux de réussite sur {n_eval_episodes} épisodes : {success_rate:.0%}")
print(f"({total_wins} victoires sur {n_eval_episodes} épisodes)")

## 10. Bonus — Visualiser la courbe d'apprentissage

On peut tracer le taux de succès au fil des épisodes pour voir comment l'agent progresse.

In [ ]:
import matplotlib.pyplot as plt

# Calculer le taux de succès glissant (fenêtre de 100 épisodes)
window = 100
smoothed = [np.mean(rewards_per_episode[max(0, i - window):i + 1]) for i in range(len(rewards_per_episode))]

plt.figure(figsize=(10, 4))
plt.plot(smoothed, color='steelblue', linewidth=1.5)
plt.xlabel("Épisode")
plt.ylabel("Taux de succès (moy. glissante 100 épisodes)")
plt.title("Courbe d'apprentissage — Q-Learning sur FrozenLake-v1")
plt.ylim(0, 1.05)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 11. Bonus — Tester avec `is_slippery=True`

Avec `is_slippery=True` (mode par défaut), l'environnement est **stochastique** : l'agent glisse et peut aller dans une direction différente de celle choisie.

- Le taux de succès sera plus faible
- Il faudra souvent plus d'épisodes d'entraînement
- Le Q-Learning converge quand même, mais la Q-table sera différente

In [ ]:
# Environnement glissant
env_slippery = gym.make('FrozenLake-v1', is_slippery=True)

q_table_slip = np.zeros((n_states, n_actions))
epsilon_slip = 1.0
n_episodes_slip = 10_000  # plus d'épisodes nécessaires

for episode in range(n_episodes_slip):
    state, info = env_slippery.reset()
    done = False

    while not done:
        if np.random.uniform(0, 1) < epsilon_slip:
            action = env_slippery.action_space.sample()
        else:
            action = np.argmax(q_table_slip[state, :])

        new_state, reward, terminated, truncated, info = env_slippery.step(action)
        done = terminated or truncated

        max_future_q = np.max(q_table_slip[new_state, :])
        target = reward + discount_factor * max_future_q
        q_table_slip[state, action] += learning_rate * (target - q_table_slip[state, action])

        state = new_state

    epsilon_slip = max(epsilon_min, epsilon_slip * epsilon_decay)

# Évaluation
wins_slip = 0
for _ in range(100):
    state, info = env_slippery.reset()
    done = False
    while not done:
        action = np.argmax(q_table_slip[state, :])
        state, reward, terminated, truncated, info = env_slippery.step(action)
        done = terminated or truncated
    if reward == 1.0:
        wins_slip += 1

print(f"Taux de réussite (is_slippery=True) : {wins_slip}%")

## Résumé

| Concept | Explication |
|---------|-------------|
| **Q-table** | Tableau `[états × actions]` qui stocke la valeur de chaque paire (état, action) |
| **Q(s, a)** | Récompense cumulée attendue si on est en état `s` et qu'on prend l'action `a` |
| **Epsilon-greedy** | Explorer aléatoirement avec probabilité ε, exploiter sinon |
| **Epsilon decay** | Réduire ε au fil du temps pour passer d'exploration à exploitation |
| **Équation de Bellman** | `Q(s,a) ← Q(s,a) + α[r + γ·maxQ(s',a') - Q(s,a)]` |
| **Alpha (α)** | Learning rate — quelle part de la nouvelle info intégrer |
| **Gamma (γ)** | Discount factor — importance des récompenses futures |
| **Erreur TD** | `r + γ·maxQ(s',a') - Q(s,a)` — écart entre cible et estimation |

### Limites du Q-Learning classique

- Fonctionne uniquement avec des espaces d'états **discrets et finis**
- Pour des espaces continus (CartPole, LunarLander), on utilise un **réseau de neurones** à la place de la Q-table → c'est le **DQN** (Deep Q-Network)

**Prochaine étape** : comprendre comment PPO et DQN étendent ces idées aux espaces continus → notebook `01_introduction_rl.ipynb`